In [114]:
import pandas as pd
import numpy as np

In [115]:
dataset = pd.read_csv("data_creator_final.csv")

In [149]:
dataset.columns

Index(['CREATOR_ID', 'VIDEO_ID', 'CREATE_TIME', 'FOLLOWERS', 'FOLLOWING_COUNT',
       'ENGAGEMENT', 'TOTAL_LIKES', 'DIGG_COUNT', 'VIDEO_COUNT',
       'COLLAB_SCORE', 'BROADCAST_SCORE', 'VIEW_COUNT', 'LIKE_COUNT',
       'COMMENT_COUNT', 'SHARE_COUNT', 'SAVE_COUNT', 'CATEGORY_TYPE',
       'CATEGORY_y', 'PRICE_NUM', 'HAS_SHOP_LINK', 'HAS_BROADCAST_SCORE'],
      dtype='object')

In [116]:
dup = dataset['VIDEO_ID'].duplicated().mean()
print("Duplicate VIDEO_ID %:", dup)

Duplicate VIDEO_ID %: 0.0


In [117]:


def audit_columns(df, cols):
    results = []

    for col in cols:
        if col not in df.columns:
            results.append({
                "column": col,
                "exists": False
            })
            continue

        series = df[col]

        results.append({
            "column": col,
            "exists": True,
            "dtype": series.dtype,
            "na_pct": series.isna().mean(),
            "zero_pct": (series == 0).mean() if np.issubdtype(series.dtype, np.number) else None,
            "neg_pct": (series < 0).mean() if np.issubdtype(series.dtype, np.number) else None,
            "median": series.median() if np.issubdtype(series.dtype, np.number) else None,
            "p90": series.quantile(0.9) if np.issubdtype(series.dtype, np.number) else None,
            "p99": series.quantile(0.99) if np.issubdtype(series.dtype, np.number) else None,
            "max": series.max() if np.issubdtype(series.dtype, np.number) else None,
        })

    return pd.DataFrame(results)


cols_to_check = [
    "LIKE_COUNT",
    "COMMENT_COUNT",
    "SHARE_COUNT",
    "SAVE_COUNT",
    "CREATE_TIME",
    "CREATOR_ID"
]

audit_report = audit_columns(dataset, cols_to_check)
print(audit_report)

          column  exists    dtype  na_pct  zero_pct  neg_pct  median      p90  \
0     LIKE_COUNT    True  float64     0.0  0.000921      0.0   626.0  13500.0   
1  COMMENT_COUNT    True    int64     0.0  0.078780      0.0    14.0    156.0   
2    SHARE_COUNT    True  float64     0.0  0.103632      0.0    20.0   1176.0   
3     SAVE_COUNT    True    int64     0.0  0.031255      0.0    41.0    902.0   
4    CREATE_TIME    True   object     0.0       NaN      NaN     NaN      NaN   
5     CREATOR_ID    True   object     0.0       NaN      NaN     NaN      NaN   

         p99        max  
0  114000.00  3600000.0  
1    1356.22   485100.0  
2   27600.00  6600000.0  
3    7714.22   295949.0  
4        NaN        NaN  
5        NaN        NaN  


In [118]:
def check_datetime(df, col="CREATE_TIME"):
    try:
        parsed = pd.to_datetime(df[col], errors='coerce')
        return {
            "total": len(parsed),
            "invalid_pct": parsed.isna().mean(),
            "min_time": parsed.min(),
            "max_time": parsed.max()
        }
    except Exception as e:
        return {"error": str(e)}

print(check_datetime(dataset))

{'total': 208479, 'invalid_pct': 0.0, 'min_time': Timestamp('2025-12-23 17:24:07'), 'max_time': Timestamp('2026-03-26 09:31:30')}


In [119]:
dataset["CREATE_TIME"] = pd.to_datetime(
    dataset["CREATE_TIME"], errors="coerce"
)

# kiểm tra
print("Invalid time %:", dataset["CREATE_TIME"].isna().mean())

Invalid time %: 0.0


In [120]:
max_time = dataset["CREATE_TIME"].max()

dataset_3m = dataset[
    dataset["CREATE_TIME"] >= (max_time - pd.DateOffset(months=3))
].copy()

In [121]:
def clean_metric(s):
    s = s.copy()
    s[s < 0] = np.nan
    return s

for col in ["LIKE_COUNT", "COMMENT_COUNT", "SHARE_COUNT", "SAVE_COUNT","VIEW_COUNT"]:
    dataset_3m[col] = clean_metric(dataset_3m[col])

In [122]:
def p90(x):
    return np.nanpercentile(x, 90)

creator_size_reach = (
    dataset_3m
    .groupby("CREATOR_ID")
    .agg(
        # ===== MEDIAN =====
        MEDIAN_LIKES_3M=("LIKE_COUNT", "median"),
        MEDIAN_COMMENTS_3M=("COMMENT_COUNT", "median"),
        MEDIAN_SHARES_3M=("SHARE_COUNT", "median"),
        MEDIAN_SAVES_3M=("SAVE_COUNT", "median"),
        MEDIAN_VIEWS_3M=("VIEW_COUNT", "median"),

        # ===== P90 =====
        P90_LIKES_3M=("LIKE_COUNT", p90),
        P90_COMMENTS_3M=("COMMENT_COUNT", p90),
        P90_SHARES_3M=("SHARE_COUNT", p90),
        P90_SAVES_3M=("SAVE_COUNT", p90),

        # ===== MAX =====
        MAX_LIKES_3M=("LIKE_COUNT", "max"),
        MAX_COMMENTS_3M=("COMMENT_COUNT", "max"),
        MAX_SHARES_3M=("SHARE_COUNT", "max"),
        MAX_SAVES_3M=("SAVE_COUNT", "max"),

        # debug
        VIDEO_COUNT_3M=("VIDEO_ID", "count")
    )
    .reset_index()
)

In [123]:
creator_size_reach.fillna(0, inplace=True)

In [124]:
print((creator_size_reach["VIDEO_COUNT_3M"] < 5).mean())
creator_size_reach.describe().to_csv("creator_size_reach_summary.csv", index=False)

0.05865823674042852


In [125]:
def safe_divide(num, denom):
    return np.where(
        (denom <= 0) | (np.isnan(denom)),
        np.nan,
        num / denom
    )

In [126]:
dataset_3m["LIKE_RATE"] = safe_divide(
    dataset_3m["LIKE_COUNT"], dataset_3m["VIEW_COUNT"]
)

dataset_3m["COMMENT_RATE"] = safe_divide(
    dataset_3m["COMMENT_COUNT"], dataset_3m["VIEW_COUNT"]
)

dataset_3m["SHARE_RATE"] = safe_divide(
    dataset_3m["SHARE_COUNT"], dataset_3m["VIEW_COUNT"]
)

dataset_3m["SAVE_RATE"] = safe_divide(
    dataset_3m["SAVE_COUNT"], dataset_3m["VIEW_COUNT"]
)



In [127]:
creator_engagement = (
    dataset_3m
    .groupby("CREATOR_ID")
    .agg(
        MEDIAN_LIKE_RATE_3M=("LIKE_RATE", "median"),
        MEDIAN_COMMENT_RATE_3M=("COMMENT_RATE", "median"),
        MEDIAN_SHARE_RATE_3M=("SHARE_RATE", "median"),
        MEDIAN_SAVE_RATE_3M=("SAVE_RATE", "median"),
    )
    .reset_index()
)

In [128]:
creator_engagement.describe()

,MEDIAN_LIKE_RATE_3M,MEDIAN_COMMENT_RATE_3M,MEDIAN_SHARE_RATE_3M,MEDIAN_SAVE_RATE_3M
count,2846.000000,2846.000000,2846.000000,2846.000000
mean,0.043313,0.001117,0.003447,0.003350
std,0.038231,0.004924,0.018588,0.005217
min,0.000375,0.000000,0.000000,0.000000
25%,0.019685,0.000294,0.000441,0.000998
50%,0.032291,0.000522,0.001041,0.001904
75%,0.054745,0.000961,0.002687,0.003862
max,0.721501,0.181939,0.903401,0.113646


In [129]:
dataset_3m["VIEW_RATE"] = safe_divide(
    dataset_3m["VIEW_COUNT"],dataset_3m["FOLLOWERS"]
)

In [130]:
creator_reach = (
    dataset_3m
    .groupby("CREATOR_ID")
    .agg(
        MEDIAN_VIEW_RATE_3M=("VIEW_RATE", "median"),
        P90_VIEW_RATE_3M=("VIEW_RATE", lambda x: np.nanpercentile(x, 90)),
        MAX_VIEW_RATE_3M=("VIEW_RATE", "max")
    )
    .reset_index()
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


In [131]:
creator_reach

,CREATOR_ID,MEDIAN_VIEW_RATE_3M,P90_VIEW_RATE_3M,MAX_VIEW_RATE_3M
0,._.ehe23,0.219885,0.600576,1.135447
1,.hoang.kha.71,0.008879,0.066596,0.780127
2,.k.c.c5,0.102294,1.552677,17.208413
3,.qkhanh1403,0.024378,0.175170,0.532451
4,.siudangiu06,0.058672,0.585419,2.152339
...,...,...,...,...
2842,ztimu_,1.939815,2.947222,3.199074
2843,zuohang2006.official,0.017127,0.136118,19.381010
2844,zybithanhtruc,0.006917,0.041444,0.118611
2845,zynanh15,0.066258,0.567261,11.692650


In [132]:
dataset_3m.groupby("CREATOR_ID")["FOLLOWERS"].nunique().value_counts()

FOLLOWERS
1    2847
Name: count, dtype: int64

In [133]:
creator_size_reach["VIDEO_COUNT_3M"].value_counts()
print((creator_size_reach["VIDEO_COUNT_3M"] < 5).mean())

0.05865823674042852


In [134]:
dataset_3m["POST_DATE"] = dataset_3m["CREATE_TIME"].dt.date

creator_time = dataset_3m.groupby("CREATOR_ID").agg(
    first_post=("CREATE_TIME", "min"),
    last_post=("CREATE_TIME", "max"),
    total_videos=("VIDEO_ID", "count"),
    unique_days=("POST_DATE", "nunique")
).reset_index()

creator_time["active_days"] = (
    (creator_time["last_post"] - creator_time["first_post"])
    .dt.days
    .clip(lower=1)
)


creator_time["VIDEOS_PER_WEEK"] = (
    creator_time["total_videos"] / creator_time["active_days"] * 7
)

creator_time["POSTING_CONSISTENCY"] = (
    creator_time["unique_days"] / (creator_time["active_days"] + 1)
)

In [142]:
max_time = dataset_3m["CREATE_TIME"].max()

creator_time["DAYS_SINCE_LAST_POST"] = (
    max_time - creator_time["last_post"]
).dt.days

In [135]:
def compute_post_gap(group):
    dates = group.sort_values().values
    if len(dates) < 2:
        return np.nan
    
    gaps = np.diff(dates).astype('timedelta64[D]').astype(int)
    return np.mean(gaps)

creator_gap = (
    dataset_3m.groupby("CREATOR_ID")["CREATE_TIME"]
    .apply(compute_post_gap)
    .reset_index(name="AVG_POST_GAP_DAYS")
)

In [136]:
creator_view = dataset_3m.groupby("CREATOR_ID").agg(
    VIEW_MEAN=("VIEW_COUNT", "mean"),
    VIEW_STD=("VIEW_COUNT", "std"),
    VIDEO_COUNT=("VIDEO_ID", "count")
).reset_index()

creator_view["VIEW_CV"] = (
    creator_view["VIEW_STD"] /
    (creator_view["VIEW_MEAN"] + 1e-6)
)

In [137]:
dataset_3m["P90_THRESHOLD"] = (
    dataset_3m.groupby("CREATOR_ID")["VIEW_COUNT"]
    .transform(lambda x: x.quantile(0.9))
)

dataset_3m["IS_VIRAL"] = (
    dataset_3m["VIEW_COUNT"] > dataset_3m["P90_THRESHOLD"]
).astype(int)

creator_virality = (
    dataset_3m.groupby("CREATOR_ID")["IS_VIRAL"]
    .mean()
    .reset_index(name="VIRALITY_INDEX")
)

In [138]:
dataset_3m["VIRAL_STRENGTH"] = (
    dataset_3m["VIEW_COUNT"] / (dataset_3m["P90_THRESHOLD"] + 1e-6)
)

creator_virality["VIRAL_INTENSITY"] = (
    dataset_3m.groupby("CREATOR_ID")["VIRAL_STRENGTH"]
    .mean()
    .values
)

In [139]:
def check_creator_key(df, name):
    if "CREATOR_ID" not in df.columns:
        print(f"[ERROR] {name}: missing CREATOR_ID")
        return
    
    n_rows = len(df)
    n_unique = df["CREATOR_ID"].nunique(dropna=False)
    n_dup = df.duplicated(subset=["CREATOR_ID"]).sum()
    
    print(f"{name}: rows={n_rows:,}, unique_creators={n_unique:,}, duplicate_keys={n_dup:,}")
    
    if n_dup > 0:
        print(df[df.duplicated(subset=["CREATOR_ID"], keep=False)]
              .sort_values("CREATOR_ID")
              .head(10))

In [143]:
for name, df in {
    "creator_size_reach": creator_size_reach,
    "creator_engagement": creator_engagement,
    "creator_time": creator_time,
    "creator_view": creator_view[["CREATOR_ID", "VIEW_MEAN", "VIEW_STD", "VIEW_CV", "VIDEO_COUNT"]],
    "creator_gap": creator_gap,
    "creator_virality": creator_virality
}.items():
    check_creator_key(df, name)

creator_size_reach: rows=2,847, unique_creators=2,847, duplicate_keys=0
creator_engagement: rows=2,847, unique_creators=2,847, duplicate_keys=0
creator_time: rows=2,847, unique_creators=2,847, duplicate_keys=0
creator_view: rows=2,847, unique_creators=2,847, duplicate_keys=0
creator_gap: rows=2,847, unique_creators=2,847, duplicate_keys=0
creator_virality: rows=2,847, unique_creators=2,847, duplicate_keys=0


In [144]:
from functools import reduce

creator_feature_tables = [
    creator_size_reach,
    creator_engagement,
    creator_time,
    creator_view[["CREATOR_ID", "VIEW_MEAN", "VIEW_STD", "VIEW_CV", "VIDEO_COUNT"]],
    creator_gap,
    creator_virality
]

creator_features = reduce(
    lambda left, right: pd.merge(
        left, right, on="CREATOR_ID", how="outer", validate="one_to_one"
    ),
    creator_feature_tables
)

In [153]:
creator_main.describe().to_csv("creator_features_summary.csv", index=False)

In [151]:
# 1) rút data gốc về đúng 1 dòng / 1 creator
creator_base = dataset.groupby("CREATOR_ID", as_index=False).first()

# 2) merge với bảng feature creator-level
creator_main = creator_base.merge(
    creator_features,
    on="CREATOR_ID",
    how="left",
    validate="one_to_one"
)

print(creator_main.shape)
print("Unique creators:", creator_main["CREATOR_ID"].nunique())
print("Duplicate CREATOR_ID:", creator_main.duplicated("CREATOR_ID").sum())

(2850, 54)
Unique creators: 2850
Duplicate CREATOR_ID: 0


In [155]:
creator_main.to_csv("creator_features.csv", index=False)